In [1]:
from datasets import load_dataset

dataset = load_dataset('thesaurus-linguae-aegyptiae/tla-Earlier_Egyptian_original-v18-premium')
print(dataset)
print(f"\nTotal sentences: {len(dataset['train'])}")
print(f"\nSample row:")
print(dataset['train'][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl:   0%|          | 0.00/5.92M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['hieroglyphs', 'transliteration', 'lemmatization', 'UPOS', 'glossing', 'translation', 'dateNotBefore', 'dateNotAfter'],
        num_rows: 12773
    })
})

Total sentences: 12773

Sample row:
{'hieroglyphs': '𓐩𓏌𓀜 𓂧 𓂋 𓋴', 'transliteration': 'nḏ (w)di̯ r =s', 'lemmatization': '90880|nḏ 51510|wdi̯ 91901|r 10090|=s', 'UPOS': 'VERB VERB ADP PRON', 'glossing': 'V\\tam.pass V\\tam.pass PREP:stpr -3sg.f', 'translation': '(es) werde zerrieben, (es) werde darauf gelegt.', 'dateNotBefore': '-1580', 'dateNotAfter': '-1539'}


In [5]:
import pandas as pd
from datasets import load_dataset

df = pd.DataFrame(dataset['train'])

# Known deity/demon names in transliteration
entity_names = {
    '(w)sꞽr': 'Osiris', 'wsꞽr': 'Osiris',
    'rꜥ': 'Ra', 'rꜥ-ḥr-ꜣḫ.tj': 'Ra-Horakhty',
    'ꞽnpw': 'Anubis',
    'ḥr': 'Horus',
    'stẖ': 'Seth', 'sṯš': 'Seth',
    'ꞽs.t': 'Isis',
    'ḏḥwtj': 'Thoth', 'ḏḥwt': 'Thoth',
    'ḥwt-ḥr': 'Hathor',
    'ptḥ': 'Ptah',
    'sḫm.t': 'Sekhmet',
    'ꞽmn': 'Amun',
    'nḫb-kꜣw': 'Nehebkau',
    'nwt': 'Nut',
    'gb': 'Geb',
    'ꜣs.t': 'Isis',
}

results = []
for idx, row in df.iterrows():
    trans = row['transliteration'].lower()
    found = {v for k, v in entity_names.items() if k in trans}
    if found:
        results.append({
            'transliteration': row['transliteration'],
            'hieroglyphs': row['hieroglyphs'],
            'entities': list(found),
            'date_from': row['dateNotBefore'],
            'date_to': row['dateNotAfter'],
            'translation': row['translation']
        })

results_df = pd.DataFrame(results)
print(f"Sentences with entities: {len(results_df)}")

from collections import Counter
all_entities = [e for row in results_df['entities'] for e in row]
for entity, count in Counter(all_entities).most_common(15):
    print(f"  {entity}: {count}")

Sentences with entities: 3896
  Horus: 2770
  Ra: 675
  Osiris: 282
  Amun: 242
  Geb: 127
  Ptah: 119
  Isis: 82
  Seth: 38
  Sekhmet: 16
  Thoth: 8
  Anubis: 7
  Nut: 2


In [4]:
for i in range(10):
      print(df['transliteration'].iloc[i])
      print()

nḏ (w)di̯ r =s

n ṯw ꞽm =sn

ḫꜣ m tʾ ḥnq.t kꜣ(.PL) ꜣpd(.PL) n ꞽmꜣḫ ꞽm.ꞽ-rʾ-šnꜥ.PL ꞽmn-m-ḥꜣ.t mꜣꜥ-ḫrw

ꜥḥꜥ

(w)sꞽr wnꞽs m n =k ꞽr.t-ḥr.w ꞽꜥb n =k s(ꞽ) ꞽr rʾ =k

ꞽwꜣ rn

qrs.t(ꞽ) m z(my).t ꞽmn.t(ꞽ).t ꞽꜣwi̯(.tꞽ) nfr wr.t (ꞽ)r(.ꞽt-ꞽ)ḫ(.t)-nswt wmt.t-kꜣ

(ꞽ)m(.ꞽ)-rʾ-zẖꜣ.w(w)-ꜥ-(n-)nswt sšm-nfr

ḥm-kꜣ mr.w-ꞽdw

ꞽḫ hnhn =s ḥr rmn.DU =ꞽ



In [6]:
from collections import Counter

all_upos = []
for row in df['UPOS']:
    all_upos.extend(row.split())

upos_counts = Counter(all_upos)
print("POS tag distribution:")
for tag, count in upos_counts.most_common():
    print(f"  {tag}: {count}")

POS tag distribution:
  NOUN: 21922
  PRON: 13435
  VERB: 11695
  PROPN: 9187
  ADP: 8080
  ADJ: 2859
  PART: 2144
  ADV: 622
  INTJ: 284
  NUM: 39


In [7]:
# Extract all PROPN tokens with their transliteration
propn_examples = []

for idx, row in df.iterrows():
    tokens = row['transliteration'].split()
    upos = row['UPOS'].split()
    lemmas = row['lemmatization'].split()

    if len(tokens) != len(upos):
        continue

    for i, (token, pos) in enumerate(zip(tokens, upos)):
        if pos == 'PROPN':
            propn_examples.append({
                'token': token,
                'sentence': row['transliteration'],
                'translation': row['translation'],
                'date': row['dateNotBefore']
            })

propn_df = pd.DataFrame(propn_examples)
print(f"Total PROPN tokens: {len(propn_df)}")
print(f"\nMost common proper nouns:")
for token, count in Counter(propn_df['token']).most_common(20):
    print(f"  {token}: {count}")

Total PROPN tokens: 9187

Most common proper nouns:
  wnꞽs: 644
  ḥr.w: 364
  ppy: 333
  n(ꞽ).t: 267
  nfr-kꜣ-rꜥw: 240
  ꜥḥꜣ: 236
  ꞽnp.w: 201
  ttꞽ: 155
  wsꞽr: 148
  rꜥw: 130
  dn: 116
  nmt.ꞽ-m-zꜣ=f: 97
  mr.n-rꜥw: 97
  gbb: 96
  ḏr: 92
  sšm-nfr: 87
  šmꜥ.w: 84
  stš: 82
  mḥ.w: 78
  nw.t: 73


In [8]:
# Known deities to separate from pharaoh names
DEITIES = {
    'ḥr.w', 'ḥr', 'wsꞽr', 'rꜥw', 'rꜥ', 'ꞽnp.w', 'gbb', 'stš', 'nw.t',
    'n(ꞽ).t', 'ꞽs.t', 'nbt-ḥw.t', 'ptḥ', 'ḏḥwtꞽ', 'ḥwt-ḥr', 'sḫm.t',
    'ꞽmn', 'ꞽmn.w', 'ꜣtm', 'nḫb.t', 'wꜣḏ.t', 'mnw', 'ḫnsw', 'mwt',
    'nḫb-kꜣw', 'sbk', 'mntw', 'sw', 'tfn.t', 'ḥḥ', 'kkw', 'nww'
}

# Build training data with character offsets for spaCy
import spacy
from spacy.tokens import DocBin
from spacy.training import Example
import re

nlp = spacy.blank("xx")  # language-agnostic blank model

training_data = []

for idx, row in df.iterrows():
    tokens = row['transliteration'].split()
    upos_tags = row['UPOS'].split()

    if len(tokens) != len(upos_tags):
        continue

    text = row['transliteration']
    entities = []

    # Find character positions of each token
    char_pos = 0
    for token, pos in zip(tokens, upos_tags):
        token_start = text.find(token, char_pos)
        if token_start == -1:
            continue
        token_end = token_start + len(token)

        if pos == 'PROPN':
            label = 'DEITY' if token.lower() in DEITIES else 'PERSON'
            entities.append((token_start, token_end, label))

        char_pos = token_end

    if entities:
        training_data.append((text, {'entities': entities}))

print(f"Training sentences with entities: {len(training_data)}")
print(f"\nSample:")
print(training_data[0])

Training sentences with entities: 7059

Sample:
('ḫꜣ m tʾ ḥnq.t kꜣ(.PL) ꜣpd(.PL) n ꞽmꜣḫ ꞽm.ꞽ-rʾ-šnꜥ.PL ꞽmn-m-ḥꜣ.t mꜣꜥ-ḫrw', {'entities': [(53, 63, 'PERSON')]})


In [9]:
from spacy.training import Example
import random

# Add NER pipe
ner = nlp.add_pipe("ner")
ner.add_label("DEITY")
ner.add_label("PERSON")

# Convert to spaCy examples
examples = []
for text, annotations in training_data:
    doc = nlp.make_doc(text)
    example = Example.from_dict(doc, annotations)
    examples.append(example)

# Train/val split
random.shuffle(examples)
train_examples = examples[:int(0.8*len(examples))]
val_examples = examples[int(0.8*len(examples)):]

print(f"Train: {len(train_examples)} | Val: {len(val_examples)}")

# Initialize and train
nlp.initialize(lambda: train_examples)

optimizer = nlp.resume_training()
best_f1 = 0
history = []

for epoch in range(20):
    random.shuffle(train_examples)
    losses = {}

    # Train in batches
    for i in range(0, len(train_examples), 32):
        batch = train_examples[i:i+32]
        nlp.update(batch, sgd=optimizer, losses=losses)

    # Evaluate
    scorer = nlp.evaluate(val_examples)
    f1 = scorer['ents_f']
    precision = scorer['ents_p']
    recall = scorer['ents_r']

    history.append({'epoch': epoch+1, 'loss': losses.get('ner', 0), 'f1': f1})

    if f1 > best_f1:
        best_f1 = f1
        nlp.to_disk('egyptian_ner_model')

    print(f"Epoch {epoch+1:02d}/20 | Loss: {losses.get('ner',0):.2f} | P: {precision:.3f} R: {recall:.3f} F1: {f1:.3f} {'* saved' if f1 == best_f1 else ''}", flush=True)

print(f"\nBest F1: {best_f1:.3f}")

Train: 5647 | Val: 1412
Epoch 01/20 | Loss: 6695.51 | P: 0.938 R: 0.906 F1: 0.921 * saved
Epoch 02/20 | Loss: 900.08 | P: 0.948 R: 0.939 F1: 0.943 * saved
Epoch 03/20 | Loss: 470.96 | P: 0.962 R: 0.923 F1: 0.942 
Epoch 04/20 | Loss: 341.75 | P: 0.956 R: 0.939 F1: 0.947 * saved
Epoch 05/20 | Loss: 285.95 | P: 0.938 R: 0.937 F1: 0.938 
Epoch 06/20 | Loss: 265.43 | P: 0.962 R: 0.935 F1: 0.949 * saved
Epoch 07/20 | Loss: 151.25 | P: 0.960 R: 0.949 F1: 0.954 * saved
Epoch 08/20 | Loss: 165.35 | P: 0.950 R: 0.952 F1: 0.951 
Epoch 09/20 | Loss: 129.78 | P: 0.954 R: 0.939 F1: 0.946 
Epoch 10/20 | Loss: 176.61 | P: 0.962 R: 0.942 F1: 0.952 
Epoch 11/20 | Loss: 156.51 | P: 0.967 R: 0.941 F1: 0.954 
Epoch 12/20 | Loss: 98.51 | P: 0.954 R: 0.949 F1: 0.951 
Epoch 13/20 | Loss: 132.32 | P: 0.968 R: 0.926 F1: 0.946 
Epoch 14/20 | Loss: 118.78 | P: 0.968 R: 0.937 F1: 0.952 
Epoch 15/20 | Loss: 86.62 | P: 0.964 R: 0.944 F1: 0.954 
Epoch 16/20 | Loss: 90.85 | P: 0.956 R: 0.950 F1: 0.953 
Epoch 17/20 | L

In [11]:
# Load best model and test
nlp_loaded = spacy.load('egyptian_ner_model')

# Test on new sentences
test_sentences = [
    "wsꞽr nb ꜣbḏw",
    "ꞽnp.w tp-ḏw=f ḥr.w nṯr ꜥꜣ",
    "dd.ꞽn rꜥw n wsꞽr",
    "stš ḫft.ꞽ n ḥr.w",
]

for sent in test_sentences:
    doc = nlp_loaded(sent)
    print(f"\nText: {sent}")
    for ent in doc.ents:
        print(f"  [{ent.label_}] {ent.text}")


Text: wsꞽr nb ꜣbḏw
  [DEITY] wsꞽr
  [PERSON] ꜣbḏw

Text: ꞽnp.w tp-ḏw=f ḥr.w nṯr ꜥꜣ
  [DEITY] ꞽnp.w
  [DEITY] ḥr.w

Text: dd.ꞽn rꜥw n wsꞽr
  [DEITY] rꜥw
  [DEITY] wsꞽr

Text: stš ḫft.ꞽ n ḥr.w
  [DEITY] stš
  [DEITY] ḥr.w


In [12]:
from google.colab import files
import shutil

shutil.make_archive('egyptian_ner_model', 'zip', 'egyptian_ner_model')
files.download('egyptian_ner_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>